<a href="https://colab.research.google.com/github/ctrlcoded/Resolving-Idioms-to-Improve-the-Machine-Translation-in-Indic-Languages/blob/main/nllb_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers[sentencepiece] sacrebleu nltk gspread pandas torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 2.3 MB/s eta 0:00:00


In [ ]:
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Google Sheets Setup
creds, _ = default()
gc = gspread.authorize(creds)
spreadsheet = gc.open('Your_Sheet_Name_Here')
worksheet = spreadsheet.get_worksheet(0)
data = worksheet.get_all_values()
df = pd.DataFrame(data[1:], columns=data[0])

# NLLB Setup
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda" if torch.cuda.is_available() else "cpu")

# NLLB uses specific language codes: 'hin_Deva' for Hindi and 'eng_Latn' for English
src_lang = "hin_Deva"
tgt_lang = "eng_Latn"

In [ ]:
from tqdm import tqdm

def translate_nllb(sentences, batch_size=16):
    results = []
    for i in tqdm(range(0, len(sentences), batch_size)):
        batch = sentences[i : i + batch_size]
        # Set source language for the tokenizer
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, src_lang=src_lang).to(model.device)

        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.lang_code_to_id[tgt_lang],
                max_length=100
            )

        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        results.extend(decoded)
    return results

# Run translation
hindi_list = df['Sentence'].tolist()
df['NLLB_Baseline_Output'] = translate_nllb(hindi_list)

In [ ]:
import sacrebleu
from nltk.translate.meteor_score import meteor_score
import nltk
nltk.download('wordnet')

def calculate_metrics(hypotheses, references, hindi_sentences):
    # BLEU
    bleu = sacrebleu.corpus_bleu(hypotheses, [references]).score
    # METEOR
    met_total = sum(meteor_score([ref.split()], hyp.split()) for ref, hyp in zip(references, hypotheses))
    avg_met = met_total / len(hypotheses)
    # LER (Keyword Spotting)
    literal_triggers = {"नाक कटना": ["nose"], "आँख का तारा": ["eye", "star"]}
    errors = 0
    total = 0
    for hi, hyp in zip(hindi_sentences, hypotheses):
        for idiom, keywords in literal_triggers.items():
            if idiom in hi:
                total += 1
                if any(word in hyp.lower() for word in keywords):
                    errors += 1
    ler = (errors / total * 100) if total > 0 else 0
    return bleu, avg_met, ler

b_bleu, b_met, b_ler = calculate_metrics(df['NLLB_Baseline_Output'].tolist(), df['Translation'].tolist(), hindi_list)

print(f"NLLB Baseline -> BLEU: {b_bleu:.2f}, METEOR: {b_met:.2f}, LER: {b_ler:.2f}%")